# Czyszczenie i przygotowanie danych

W tym notatniku przeprowadzimy pełny proces czyszczenia danych ze zbioru `data/sklep_rowerowy.csv`. Zidentyfikujemy brakujące wartości, sprawdzimy występowanie duplikatów, przeanalizujemy typy danych, zdiagnozujemy wartości odstające (outliery) oraz dokonamy imputacji braków za pomocą odpowiednich narzędzi.

### Wczytanie bibliotek
Rozpoczynamy od importu bibliotek `pandas` oraz `numpy`, które potrzebujemy do uporządkowania i czyszczenia danych.

In [ ]:
import pandas as pd
import numpy as np

### Wczytanie danych
Wczytujemy plik CSV o nazwie `data/sklep_rowerowy.csv` do ramki danych biblioteki pandas (`DataFrame`) i wyświetlamy podgląd pierwszych pięciu wierszy oraz rozmiar zbioru.

In [ ]:
df = pd.read_csv('data/sklep_rowerowy.csv')
print(f"Rozmiar zbioru: {df.shape[0]} wierszy, {df.shape[1]} kolumn.\n")
df.head(5)

Następnie, możemy sprawdzić informacje o naszych danych.

In [ ]:
df.info()

### Konwersja typów na całkowite (int) oraz tekstowe (string)
Po wczytaniu możemy zmienić kolumny na odpowiednie typy docelowe. Kolumny numeryczne (`Income`, `Children`, `Cars`, `Age`) konwertujemy na typ całkowitoliczbowy (`Int64`), a kolumnę `ID` oraz pozostałe kolumny tekstowe na typ tekstowy (`string`). Robimy to, bo typy te w pandas obsługują wartości brakujące (NaN).

In [ ]:
df['ID'] = df['ID'].astype('string')

cols_to_int = ['Income', 'Children', 'Cars', 'Age']
for col in cols_to_int:
    df[col] = df[col].astype('Int64')

categorical_cols = ['Marital Status', 'Gender', 'Education', 'Occupation', 'Home Owner', 'Commute Distance', 'Region', 'Purchased Bike']
for col in categorical_cols:
    df[col] = df[col].astype('string')

print("--- Typy danych kolumn po konwersji ---")
print(df.dtypes.to_string())
df.head()

### Puste komórki
Sprawdzamy, ile brakujących wartości (`NaN`/`null`) znajduje się w każdej kolumnie, aby dowiedzieć się, które kolumny wymagają imputacji.

In [ ]:
missing_values = df.isnull().sum()
print("Liczba braków w poszczególnych kolumnach:")
print(missing_values)

### Duplikaty
Sprawdzamy, czy w naszym zbiorze występują zduplikowane wiersze oraz czy identyfikatory klientów (`ID`) są unikalne.

In [ ]:
total_duplicates = df.duplicated().sum()
id_duplicates = df['ID'].duplicated().sum()
print(f"Liczba całkowicie zduplikowanych wierszy: {total_duplicates}")
print(f"Liczba zduplikowanych identyfikatorów ID: {id_duplicates}")

### Sprawdzenie unikalnych wartości i jeszcze raz (na wszelki wypadek) typów kolumn
Przed modyfikacją danych sprawdzamy typy danych poszczególnych kolumn oraz unikalne wartości dla kolumn tekstowych (kategorycznych). Pozwoli to zlokalizować ewentualne ukryte problemy (np. spacje w tekście czy inne nietypowe wartości).

In [ ]:
print("--- Typy danych kolumn ---")
print(df.dtypes)

print("\n--- Unikalne wartości w kolumnach kategorycznych ---")
categorical_cols = ['Marital Status', 'Gender', 'Education', 'Occupation', 'Home Owner', 'Commute Distance', 'Region', 'Purchased Bike']
for col in categorical_cols:
    print(f"\nKolumna '{col}': {df[col].unique()}")

### Diagnostyka wartości odstających
Sprawdzamy, czy w kolumnach numerycznych (`Income`, `Children`, `Cars`, `Age`) występują wartości odstające (outliery). Użyjemy do tego metody IQR (rozstęp ćwiartkowy), wyświetlając medianę, logiczne granice zakresu typowego (zabezpieczone przed wartościami ujemnymi) oraz 5 najbardziej odstających wartości.

In [ ]:
numerical_cols = ['Income', 'Children', 'Cars', 'Age']
print("--- Diagnostyka wartości odstających (metoda IQR) ---")
for col in numerical_cols:
    # Obliczamy kwartyle i rozstęp ćwiartkowy (IQR) dla niepustych wartości
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Określamy teoretyczne granice wartości odstających
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Ponieważ dochód, dzieci, samochody nie mogą być ujemne, dolną granicę dla celów prezentacyjnych ograniczamy do 0
    logical_lower_bound = max(0, lower_bound)
    
    # Filtrujemy outliery z wykorzystaniem matematycznych granic
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].copy()
    median_val = df[col].median()
    
    print(f"\nKolumna '{col}':")
    print(f"  Mediana: {median_val}")
    print(f"  Zakres typowy (IQR): [{logical_lower_bound}, {upper_bound}]")
    print(f"  Liczba zidentyfikowanych outlierów: {len(outliers)}")
    
    if len(outliers) > 0:
        # Obliczamy odległość od mediany i sortujemy malejąco
        outliers['distance'] = (outliers[col] - median_val).abs()
        top_outliers = outliers.sort_values(by='distance', ascending=False).head(5)
        print("  5 najbardziej odstających wartości:")
        for idx, row in top_outliers.iterrows():
            val = row[col]
            print(f"    Wartość: {val} (odległość od mediany: {abs(val - median_val)})")

#### Co robimy z wartościami odstającymi?
Po analizie wyników identyfikacji wartości odstających podejmujemy następujące decyzje:
1. **Wysokie zarobki (Income):** Kilka osób posiada dochód powyżej górnej granicy (np. 160 000 - 170 000 USD). Nie są to jednak błędy w danych, lecz naturalna grupa zamożniejszych klientów. **Decyzja: pozostawiamy te dane bez zmian.**
2. **Wiek (Age):** Maksymalny wiek w zbiorze to 89 lat, co również jest realną wartością i nie stanowi błędu systemowego. **Decyzja: pozostawiamy te dane bez zmian.**
3. **Liczba dzieci (Children) i samochodów (Cars):** Kolumny te nie wykazują anomalii matematycznych ani wartości wykraczających poza logiczny zakres.
4. **Podsumowanie:** Wszystkie zidentyfikowane matematycznie outliery reprezentują rzeczywiste i poprawne scenariusze życiowe klientów. Ich usunięcie zniekształciłoby późniejszą analizę segmentacji klientów. Dlatego **decydujemy się pozostawić je w zbiorze danych**, a przy analizie statystycznej i wizualizacji (np. wykresy pudełkowe) będziemy pamiętać o ich obecności.

### Podstawowa obróbka danych – usuwanie spacji
Stosujemy metodę `.str.strip()` do wszystkich kolumn tekstowych (typu `string`), aby pozbyć się zbędnych spacji na początku i na końcu ciągów znaków. Metoda ta automatycznie ignoruje wartości brakujące.

In [ ]:
for col in df.select_dtypes(include=['string']).columns:
    df[col] = df[col].str.strip()
print("Usunięto zbędne spacje z kolumn tekstowych.")

### Imputacja brakujących wartości
Przechodzimy do uzupełnienia braków danych:
1. Kolumny jakościowe (kategoryczne: `Marital Status`, `Gender`, `Home Owner`) uzupełniamy **modą** (najczęściej występującą wartością).
2. Kolumny ilościowe (numeryczne: `Income`, `Children`, `Cars`, `Age`) uzupełniamy **medianą** (wartością środkową), co chroni nas przed wpływem ewentualnych wartości odstających.

In [ ]:
df_before = df.copy()

# Imputacja kolumn kategorycznych za pomocą mody
cat_to_impute = ['Marital Status', 'Gender', 'Home Owner']
for col in cat_to_impute:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"Kolumna '{col}': uzupełniono modą -> '{mode_val}'")

# Imputacja kolumn numerycznych za pomocą mediany
num_to_impute = ['Income', 'Children', 'Cars', 'Age']
for col in num_to_impute:
    
    median_val = int(df[col].median())
    df[col] = df[col].fillna(median_val)
    print(f"Kolumna '{col}': uzupełniono medianą -> {median_val}")

### Weryfikacja braków po imputacji
Porównujemy liczbę brakujących komórek w każdej kolumnie przed i po przeprowadzeniu procesu imputacji, aby upewnić się, że w zbiorze nie ma już żadnych braków danych.

In [ ]:
comparison_df = pd.DataFrame({
    'Przed imputacją': df_before.isnull().sum(),
    'Po imputacji': df.isnull().sum()
})
print("Tabela porównawcza liczby braków danych:")
print(comparison_df)

### Zapisanie oczyszczonych danych do nowego pliku
Na zakończenie procesu zapisujemy oczyszczoną ramkę danych do nowego pliku o nazwie `data/sklep_rowerowy_oczyszczone.csv`. W ten sposób zachowujemy oryginalny plik bez zmian, a oczyszczone dane są gotowe do dalszej analizy.

In [ ]:
output_file = 'data/sklep_rowerowy_oczyszczone.csv'
df.to_csv(output_file, index=False)
print(f"Oczyszczone dane zostały pomyślnie zapisane do pliku: {output_file}")

### Raport Skimpy

In [1]:
from skimpy import skim
print("\n--- Wywołanie raportu strukturalnego skimpy ---")
skim(df)

ModuleNotFoundError: No module named 'skimpy'